Step 1: Create a GitHub Personal Access Token

In [1]:
import requests
import os

GITHUB_TOKEN = "your_github_token"
REPO_OWNER = "your_github_username"
REPO_NAME = "repository_name"

Step 2: Environment Setup and Authentication

In [20]:
headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

Step 3: Fetching Pull Requests and Code Diffs

Goal - find open pull requests and get the specific files that have changed

In [27]:
def get_pull_requests():
    # Endpoint to list pull requests
    url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/pulls"
    response = requests.get(url, headers=headers)
    return response.json()

def get_pr_files(pr_number):
    # Endpoint to get the specific files changed in a PR
    url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/pulls/{pr_number}/files"
    response = requests.get(url, headers=headers)
    return response.json()

# This function returns the patch data, showing the lines of code that were added or removed. 

Step 4: Analyzing Code with a Local LLM

In [30]:
import json

def analyze_code(code_patch):

    # Define the system prompt and instructions
    prompt = f"""
You are an expert software engineer performing a code review.

Analyze the following code changes and suggest improvements,
possible bugs, performance issues, and style improvements.

Code Changes:
{code_patch}

Provide clear suggestions.
"""

    # Send the prompt to our local Ollama instance running Mistral
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "mistral",
            "prompt": prompt,
            "stream": False
        }
    )

    result = response.json()
    return result["response"]

Step 5:  Posting the Review back to GitHub

In [33]:
def post_comment(pr_number, comment):

    # GitHub API treats PR comments as issue comments
    url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/issues/{pr_number}/comments"

    data = {
        "body": f"AI Code Review Suggestions:\n\n{comment}"
    }

    requests.post(url, headers=headers, json=data)